In [1]:
from datetime import datetime, date
import pandas as pd
import itertools
from finrl.agents.elegantrl.models import DRLAgent
from enum import StrEnum, auto
import numpy as np
from torch import optim
import gymnasium as gym

from matplotlib import pyplot as plt
import tempfile
import ray
from ray import train
from torch.utils.tensorboard import SummaryWriter
from ray.train import Checkpoint
from ray import tune
from ray.air import session
from ray.tune import CLIReporter
from ray.tune.schedulers import ASHAScheduler
from finrl.meta.data_processor import DataProcessor
from finrl.meta.preprocessor.yahoodownloader import YahooDownloader
from finrl.meta.preprocessor.preprocessors import FeatureEngineer
from stable_baselines3.common.logger import configure
from finrl.meta.env_stock_trading.env_forex_price_trailing import ForexPriceTrailingEnv
import torch as th
import torch
import torch.nn as nn
from collections import deque
import random
import os
import torch.nn.functional as F
from elegantrl.train.config import Config
from elegantrl.train.run    import train_agent
import torch.nn as nns
from elegantrl.agents import AgentDQN, AgentDoubleDQN
from copy import deepcopy
%matplotlib inline

In [2]:
dash_writer = SummaryWriter("runs/price_trail_experiment/ddqn")

In [3]:
class Currency(StrEnum):
    USD = auto()
    EUR = auto()
    JPY = auto()
    GBP = auto()
    AUD = auto()
    CAD = auto()
    CHF = auto()
    NZD = auto()
    CNY = auto()

In [4]:
def max_drawdown(returns: np.ndarray) -> float:
    cum = np.cumsum(returns)
    peak = np.maximum.accumulate(cum)
    drawdowns = cum - peak
    return drawdowns.min()

In [5]:
def add_fx_features_for_tick(g: pd.DataFrame) -> pd.DataFrame:
    g["close_prev"] = g.close.shift(1)
    g["high_prev"] = g.high.shift(1)
    g["low_prev"] = g.low.shift(1)

    g["x1"] = (g.close - g.close_prev) / g.close_prev
    g["x2"] = (g.high - g.high_prev) / g.high_prev
    g["x3"] = (g.low - g.low_prev) / g.low_prev
    g["x4"] = (g.high - g.close) / g.close
    g["x5"] = (g.close - g.low) / g.close
    return g

def add_fx_features(df: pd.DataFrame, tic_col: str = "tic") -> pd.DataFrame:
    df_with_features = (
        df
        .groupby(tic_col, group_keys=False)
        .apply(add_fx_features_for_tick)
        .drop(columns=["close_prev", "high_prev", "low_prev"])
        .fillna(0)
    )
    return df_with_features

In [6]:
class LSTM_QNet(nn.Module):
    def __init__(
        self,
        window:      int = 16,
        feature_dim: int = 5,
        lstm_hidden: int = 32,
        fc_hidden:   int = 64,
        action_dim:  int = 3
    ):
        super().__init__()
        self.window      = window
        self.feature_dim = feature_dim
        self.action_dim  = action_dim

        # LSTM over (window × feature_dim)
        self.lstm = nn.LSTM(input_size=feature_dim,
                            hidden_size=lstm_hidden,
                            batch_first=True)
        # small FC on top of LSTM
        self.fc_after_lstm = nn.Linear(lstm_hidden, lstm_hidden)
        # two FC layers after concatenating prev-pos one-hot
        self.fc1 = nn.Linear(lstm_hidden + action_dim, fc_hidden)
        self.fc2 = nn.Linear(fc_hidden,       fc_hidden)
        # final head: Q-values for each action
        self.q_head = nn.Linear(fc_hidden, action_dim)

    def get_q_value(self, state: torch.Tensor) -> torch.Tensor:
        """
        state: (B, window*feature_dim + 1)
        returns Q(s,·): (B, action_dim)
        """
        B = state.size(0)
        hist = state[:, : self.window*self.feature_dim]
        prev = state[:, -1].long()               # δ ∈ {-1,0,1}
        seq  = hist.view(B, self.window, self.feature_dim)

        lstm_out, _ = self.lstm(seq)             # (B, window, lstm_hidden)
        h_T         = lstm_out[:, -1, :]         # (B, lstm_hidden)
        h           = F.relu(self.fc_after_lstm(h_T))

        oh          = F.one_hot(prev+1, self.action_dim).float()  # (B,3)
        z           = torch.cat([h, oh], dim=1)                  # (B, lstm+3)
        z1          = F.relu(self.fc1(z))
        z2          = F.relu(self.fc2(z1))
        q           = self.q_head(z2)                             # (B,3)
        return q

    def forward(self, state: torch.Tensor) -> torch.Tensor:
        """
        called during inference: returns greedy action as (B,1)
        """
        return self.get_q_value(state).argmax(dim=1, keepdim=True)

# ─── 3) REPLAY BUFFER ───────────────────────────────────────────────────────────

class ReplayBuffer:
    def __init__(self, capacity):
        self.capacity = capacity
        self.buffer   = deque(maxlen=capacity)

    def push(self, s, a, r, s2, done):
        self.buffer.append((s, a, r, s2, done))

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        s,a,r,s2,d = map(np.stack, zip(*batch))
        return (
            torch.FloatTensor(s),
            torch.LongTensor(a),
            torch.FloatTensor(r),
            torch.FloatTensor(s2),
            torch.FloatTensor(1 - d)
        )

    def __len__(self):
        return len(self.buffer)

In [7]:
def plot_performance(history: dict[str, list], title: str):
    fig, (ax_price, ax_r) = plt.subplots(2,1,figsize=(12,6), sharex=True)
    if title:
        fig.suptitle(title, fontsize=16, y=0.98)  

    ax_price.plot(history["date"], history["close"],      label="Close Price")
    ax_price.plot(history["date"], history["agent"],      linestyle="--", label="Agent Price")
    ax_price.plot(history["date"], history["upper"],      color="gray", alpha=0.5, label="Upper Bound")
    ax_price.plot(history["date"], history["lower"],      color="gray", alpha=0.5, label="Lower Bound")
    ax_price.set_ylabel("Price")
    ax_price.legend(loc="best")

    ax_r.plot(history["date"], history["reward"], label="Shaped Reward")
    ax_r.set_ylabel("Reward")
    ax_r.set_xlabel("Date")
    ax_r.legend(loc="best")

    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

In [8]:
def evaluate_greedy(
    net: nn.Module, 
    env: ForexPriceTrailingEnv, 
    repeats: int = 5,
    global_step: int = 0
) -> tuple[float, float, float]:
    pnls = []
    all_step_returns = []
    last_pnl_series = None

    for _ in range(repeats):
        env.reset()
        env.set_fixed_window(env.start_idx, env.EP_LEN)
        done = False
        trunc = False
        obs, _ = env.reset()
        pnl_series = []

        while not (done or trunc):
            with torch.no_grad():
                action = net(torch.tensor(obs, dtype=torch.float32).unsqueeze(0)).item()
            obs, _, done, trunc, info = env.step(int(action))
            pnl_series.append(info.get("pnl", 0.0))
        pnls.append(np.sum(pnl_series))
        last_pnl_series = pnl_series  

    returns = np.array(pnls)
    sharpe = returns.mean() / (returns.std() + 1e-9) * np.sqrt(252 * 24)
    mdd = max_drawdown(returns)
    mean_ret = returns.mean()

    dash_writer.add_scalar("eval/Sharpe", sharpe, global_step)
    dash_writer.add_scalar("eval/MeanReturn", mean_ret, global_step)
    dash_writer.add_scalar("eval/MaxDrawdown", mdd, global_step)
    fig, axes = plt.subplots(2, 2, figsize=(10, 8))
    
    # a) Cumulative PnL for last episode
    cum = np.cumsum(last_pnl_series)
    axes[0,0].plot(cum)
    axes[0,0].set_title("Last Episode: Cumulative PnL")
    axes[0,0].set_xlabel("Step")
    axes[0,0].set_ylabel("Cum. PnL")

    # b) Drawdown curve
    peak = np.maximum.accumulate(cum)
    dd = (cum - peak) / (peak + 1e-9)
    axes[0,1].plot(dd)
    axes[0,1].set_title("Last Episode: Drawdown")
    axes[0,1].set_xlabel("Step")
    axes[0,1].set_ylabel("Drawdown")

    # c) Histogram of total PnLs
    axes[1,0].hist(returns, bins=20, edgecolor='k')
    axes[1,0].set_title("Distribution of Total PnL (repeats)")
    axes[1,0].set_xlabel("Total PnL")
    axes[1,0].set_ylabel("Frequency")

    # d) Boxplot of all step-returns
    axes[1,1].boxplot(all_step_returns, vert=False)
    axes[1,1].set_title("Boxplot of Step Returns")
    axes[1,1].set_xlabel("Step Return")

    plt.tight_layout()

    # 3) Push that grid into TB
    dash_writer.add_figure(f"eval_step_{global_step}/EvaluationGrid", fig)
    plt.close(fig)
    return sharpe, returns.mean(), mdd

In [9]:
def train_ddqn(
    env_train: ForexPriceTrailingEnv,
    env_eval: ForexPriceTrailingEnv,
    q_net: nn.Module,
    target_q: nn.Module,
    buffer,
    episodes: int = 5000,
    eval_every: int = 50,
    patience: int = 8,
    min_delta: float = 0.01,
    batch_size: int = 512,
    gamma: float = 0.995,
    lr: float = 1e-4,
    eps_start: float = 1.0,
    eps_end: float = 0.01,
    eps_decay: float = 25000,
    target_update: int = 10
):
    optim_q = optim.Adam(q_net.parameters(), lr=lr)
    steps_done = 0
    best_sharpe = -np.inf
    wait = 0
    best_state = deepcopy(q_net.state_dict())

    for ep in range(1, episodes + 1):
        obs, _ = env_train.reset()
        ep_reward = 0.0

        for t in range(env_train.EP_LEN):
            eps = eps_end + (eps_start - eps_end) * np.exp(-1. * steps_done / eps_decay)
            dash_writer.add_scalar("epsilon", eps, (ep - 1) * env_train.EP_LEN + t)
            steps_done += 1

            if random.random() > eps:
                with torch.no_grad():
                    action = q_net(torch.tensor(obs, dtype=torch.float32).unsqueeze(0)).item()
            else:
                action = env_train.action_space.sample()
            
            next_obs, reward, done, trunc, info = env_train.step(int(action))
            ep_reward += reward
            buffer.push(obs, action, reward, next_obs, done or trunc)
            obs = next_obs

            if len(buffer) >= batch_size:
                s_batch, a_batch, r_batch, s2_batch, not_done = buffer.sample(batch_size)

                q_vals = q_net.get_q_value(s_batch)
                q_a = q_vals.gather(1, a_batch.unsqueeze(1))

                with torch.no_grad():
                    next_actions = q_net.get_q_value(s2_batch).argmax(dim=1, keepdim=True)
                    q2 = target_q.get_q_value(s2_batch)
                    q2_a = q2.gather(1, next_actions)
                    q_target = r_batch.unsqueeze(1) + gamma * not_done.unsqueeze(1) * q2_a

                loss = F.smooth_l1_loss(q_a, q_target)
                optim_q.zero_grad()
                loss.backward()
                optim_q.step()

            if done or trunc:
                break

        if ep % target_update == 0:
            target_q.load_state_dict(q_net.state_dict())
        
        if ep % eval_every == 0:
            sharpe, avg_pnl, mdd = evaluate_greedy(q_net, env_eval, 5, steps_done)
            print(f"Eval @ Ep {ep}: Sharpe={sharpe:.3f}, PnL={avg_pnl:.2f}, MDD={mdd:.2f}")
            if sharpe > best_sharpe + min_delta:
                best_sharpe = sharpe
                wait = 0
                best_state = deepcopy(q_net.state_dict())
            else:
                wait += 1
            if wait >= patience:
                print(f"Early stopping at ep {ep}, best Sharpe={best_sharpe:.3f}")
                q_net.load_state_dict(best_state)
                break
    return q_net

## Data preprocessing

In [10]:
majors = [
    "EURUSD=X","USDJPY=X","GBPUSD=X",
    "AUDUSD=X","USDCAD=X","USDCHF=X","NZDUSD=X"
]

# 2) Top non-USD crosses
crosses = [
    "EURGBP=X","EURJPY=X","GBPJPY=X","AUDJPY=X",
    "CADJPY=X","EURAUD=X","EURCAD=X","EURCHF=X",
    "GBPCHF=X","AUDCAD=X","NZDJPY=X","NZDCAD=X"
]

# 3) Key CNY pairs
cny_pairs = [
    "USDCNY=X","EURCNY=X","JPY CNY=X".replace(" ",""),  # -> "JPYCNY=X"
    "GBPCNY=X","AUDCNY=X","CADCNY=X","CHFCNY=X","NZDCNY=X"
]

# 4) Stitch together, then take the first 20 unique
all_tickers = majors + crosses + cny_pairs
# remove any duplicates and slice to 20
seen = set()
forex_ticks: tuple[str, ...] = tuple(
    t for t in all_tickers
    if not (t in seen or seen.add(t))
)
len(forex_ticks)

27

In [11]:
start_train = date(2000,1,1)
end_train = date(2017,1,1)
eval_span = (date(2016,7,1), date(2017,1,1))  # last 6 months
test_span = (date(2017,1,1), date(2018,6,1))

In [12]:
yfd = YahooDownloader(start_date=str(start_train), end_date=str(end_train), ticker_list=forex_ticks)
df_train = add_fx_features(yfd.fetch_data())
df_train

YF deprecation warning: set proxy via new config function: yf.set_config(proxy=proxy)


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%********

Shape of DataFrame:  (93926, 8)


/var/folders/23/n0prkghs6v94ssbsv_9xq74c0000gn/T/ipykernel_17093/2277383529.py:17: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(add_fx_features_for_tick)


Price,date,close,high,low,open,volume,tic,day,x1,x2,x3,x4,x5
0,2000-01-03,0.625400,0.629000,0.620300,0.623900,0,EURGBP=X,0,0.000000,0.000000,0.000000,0.005756,0.008155
1,2000-01-03,101.690002,103.330002,101.309998,102.070000,0,USDJPY=X,0,0.000000,0.000000,0.000000,0.016127,0.003737
2,2000-01-04,0.628300,0.631400,0.623600,0.625300,0,EURGBP=X,1,0.004637,0.003816,0.005320,0.004934,0.007481
3,2000-01-04,103.139999,103.320000,101.470001,101.639999,0,USDJPY=X,1,0.014259,-0.000097,0.001579,0.001745,0.016192
4,2000-01-05,0.628400,0.633000,0.626800,0.628100,0,EURGBP=X,2,0.000159,0.002534,0.005131,0.007320,0.002546
...,...,...,...,...,...,...,...,...,...,...,...,...,...
93921,2016-12-30,0.697204,0.697496,0.694493,0.697204,0,NZDUSD=X,4,0.006484,0.000488,0.003820,0.000418,0.003889
93922,2016-12-30,1.347810,1.349390,1.340240,1.347760,0,USDCAD=X,4,-0.005497,-0.004632,-0.005963,0.001172,0.005617
93923,2016-12-30,1.017100,1.022820,1.014400,1.016920,0,USDCHF=X,4,-0.010757,-0.005271,-0.007825,0.005624,0.002655
93924,2016-12-30,6.945300,6.954600,6.934500,6.954600,0,USDCNY=X,4,-0.001840,-0.000546,-0.001454,0.001339,0.001555


In [13]:
yfd2 = YahooDownloader(start_date=str(test_span[0]), end_date=str(test_span[1]), ticker_list=forex_ticks)
df_test = add_fx_features(yfd2.fetch_data())
df_test

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%********

Shape of DataFrame:  (9915, 8)



/var/folders/23/n0prkghs6v94ssbsv_9xq74c0000gn/T/ipykernel_17093/2277383529.py:17: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(add_fx_features_for_tick)


Price,date,close,high,low,open,volume,tic,day,x1,x2,x3,x4,x5
0,2017-01-02,0.964940,0.970550,0.960000,0.964940,0,AUDCAD=X,0,0.000000,0.000000,0.000000,0.005814,0.005120
1,2017-01-02,4.980900,5.002400,4.972400,4.980900,0,AUDCNY=X,0,0.000000,0.000000,0.000000,0.004317,0.001706
2,2017-01-02,84.081001,84.421997,83.959000,84.309998,0,AUDJPY=X,0,0.000000,0.000000,0.000000,0.004056,0.001451
3,2017-01-02,0.720981,0.722178,0.717000,0.720981,0,AUDUSD=X,0,0.000000,0.000000,0.000000,0.001661,0.005521
4,2017-01-02,5.144400,5.184700,5.143300,5.144400,0,CADCNY=X,0,0.000000,0.000000,0.000000,0.007834,0.000214
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9910,2018-05-31,0.698461,0.702395,0.697078,0.698592,0,NZDUSD=X,3,0.013816,0.004081,0.012032,0.005633,0.001980
9911,2018-05-31,1.289310,1.298810,1.281900,1.289160,0,USDCAD=X,3,-0.010507,-0.003888,-0.001503,0.007368,0.005747
9912,2018-05-31,0.988580,0.988900,0.982500,0.988560,0,USDCHF=X,3,-0.002140,-0.004580,-0.005516,0.000324,0.006150
9913,2018-05-31,6.418000,6.418000,6.394800,6.409000,0,USDCNY=X,3,0.000218,-0.002270,-0.001998,0.000000,0.003615


In [14]:
sample_df = df_train[df_train.tic == forex_ticks[0]].reset_index(drop=True)
val_start_idx = sample_df[pd.to_datetime(sample_df.date) == pd.Timestamp(eval_span[0])].index[0]
val_length = int((eval_span[1] - eval_span[0]).days)
val_start_idx

np.int64(3258)

In [15]:
window = 16
env_context = {
    "tic_col": "tic",
    "window": 16,
    "margin": 0.02,
    "step_frac": 0.1,
    "fee": 2e-4,
    "alpha_trail": 0.97,
    "alpha_pnl": 0.85,
    "alpha_fee": 1.0,
    "episode_len": 600,
    "pick_new_pair_every": 1,
}
train_env = ForexPriceTrailingEnv(df_train, **env_context)
eval_env_context = env_context
eval_env_context["episode_len"] = 150
eval_env   = ForexPriceTrailingEnv(df_train, **eval_env_context)
eval_env.set_fixed_window(val_start_idx, val_length)
test_env = ForexPriceTrailingEnv(df_test, window=window)

In [16]:
q_net = LSTM_QNet(
    window=window,
    feature_dim=5,
    lstm_hidden=128,
    fc_hidden=64,
    action_dim=3
)
target_q = deepcopy(q_net)
buffer = ReplayBuffer(capacity=200_000)

In [17]:
best_model = train_ddqn(
    train_env,
    eval_env,
    q_net,
    target_q,
    buffer,
    episodes=2000,
    eval_every=50,
    patience=8,
    batch_size=512,
    eps_decay=25000
)

Picking new FX pair: JPYCNY=X (episode #1)
Picking new FX pair: GBPUSD=X (episode #2)
Picking new FX pair: EURCHF=X (episode #3)
Picking new FX pair: GBPUSD=X (episode #4)
Picking new FX pair: USDCAD=X (episode #5)
Picking new FX pair: AUDJPY=X (episode #6)
Picking new FX pair: NZDJPY=X (episode #7)
Picking new FX pair: EURAUD=X (episode #8)
Picking new FX pair: CADJPY=X (episode #9)
Picking new FX pair: EURCHF=X (episode #10)
Picking new FX pair: EURAUD=X (episode #11)
Picking new FX pair: NZDUSD=X (episode #12)
Picking new FX pair: EURGBP=X (episode #13)
Picking new FX pair: EURCNY=X (episode #14)
Picking new FX pair: EURGBP=X (episode #15)
Picking new FX pair: AUDCAD=X (episode #16)
Picking new FX pair: EURCAD=X (episode #17)
Picking new FX pair: CADCNY=X (episode #18)
Picking new FX pair: NZDCNY=X (episode #19)
Picking new FX pair: AUDCNY=X (episode #20)
Picking new FX pair: USDCHF=X (episode #21)
Picking new FX pair: EURJPY=X (episode #22)
Picking new FX pair: AUDCAD=X (episode #2

KeyboardInterrupt: 

In [ ]:
sharpe_test, pnl_test, mdd_test = evaluate_greedy(best_model, test_env)
print(f"Test results: Sharpe={sharpe_test:.3f}, PnL={pnl_test:.2f}, MDD={mdd_test:.2f}")

In [ ]:
# window = 16 
# env = ForexPriceTrailingEnv(
#     full_df=df_features,
#     window=window,
#     margin=0.02,
#     step_frac=0.1,
#     episode_len=2000,
# )

# q_net, target_q, episode_rewards = train_ddqn(
#     env=env,
#     q_net=LSTM_QNet(window=window, feature_dim=5, lstm_hidden=128, fc_hidden=64, action_dim=3),
#     target_q=LSTM_QNet(window=window, feature_dim=5, lstm_hidden=128, fc_hidden=64, action_dim=3),
#     buffer=ReplayBuffer(capacity=100_000),
#     episodes=50,
#     max_steps=2000,
#     batch_size=512,
#     gamma=0.995,
#     lr=1e-4,
#     eps_start=1.0,
#     eps_end=0.01,
#     eps_decay=25_000,
#     target_update=10
# )